# Data preprocessing

Jupyter Notebook containing the data preprocessing, molecular standardization, Morgan fingerprint generation, and machine learning analysis of the Tox21 dataset. Includes training and evaluation of Logistic Regression and Random Forest models for SR-ARE activity prediction.


In [5]:
import pandas as pd


In [6]:
import numpy as np

In [7]:
df=pd.read_csv("../data/tox21.csv")

In [8]:
print("raw dataset shape:", df.shape)

raw dataset shape: (7831, 14)


In [9]:
print("columns: \n")
print(df.columns.tolist())

columns: 

['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']



Target columns NR-AR, NR-ER, SR-ARE, etc. — toxicology test results;
we store them in `label_cols`

In [10]:
label_cols = [
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53"
]

Convert target columns to numeric values

In [11]:
df[label_cols] = df[label_cols].apply(pd.to_numeric, errors = "coerce")

In [12]:
print("number of rows:", len(df))

number of rows: 7831


In [13]:
print(" missing values per column:")

 missing values per column:


In [14]:
print(df.isna().sum())

NR-AR             566
NR-AR-LBD        1073
NR-AhR           1282
NR-Aromatase     2010
NR-ER            1638
NR-ER-LBD         876
NR-PPAR-gamma    1381
SR-ARE           1999
SR-ATAD5          759
SR-HSE           1364
SR-MMP           2021
SR-p53           1057
mol_id              0
smiles              0
dtype: int64


In [15]:
print("SR-Are value counts: \n")
print(df["SR-ARE"].value_counts(dropna=False))

SR-Are value counts: 

SR-ARE
0.0    4890
NaN    1999
1.0     942
Name: count, dtype: int64


In [16]:
from rdkit import RDLogger

In [17]:
RDLogger.DisableLog("rdApp.*")

In [18]:
from rdkit import Chem

This function is used to convert a SMILES string into an RDKit chemical structure—a `Mol` object.
RDKit takes a text-based SMILES string:

CCO

and converts it into a special molecule object:

SMILES
  ↓
“CCO”
  ↓
Chem.MolFromSmiles()
  ↓
RDKit Mol object

RDKit recognizes the following in it:

atoms;
bonds;
bond types;
rings;
charges;
chemical structure.

In [19]:


def to_mol(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    
    return Chem.MolFromSmiles(smiles)   

Calling the function to create a new column containing molecular objects.

In [20]:
df["mol"] =df["smiles"].apply(to_mol)

This is a counter variable that tracks how many None/NaN values there are.

In [21]:
invalid_smiles = df["mol"].isna().sum()

In [22]:
print("invalid smiles:", invalid_smiles)

invalid smiles: 8


removing invalid smiles

In [23]:
df=df[df["mol"].notna()].copy()

In [24]:
print(" rows after removing invalid smiles: ", len(df))

 rows after removing invalid smiles:  7823


This step involves deriving the canonical SMILES. Its purpose is to convert descriptions of the same molecule into a single, standardized text representation, making it easier to detect duplicates later on.

In [25]:
def mol_to_canonical_smiles(mol) :
    return Chem.MolToSmiles(mol, canonical=True)

In [26]:
df["smiles_std"]  =df["mol"].apply(mol_to_canonical_smiles)

In [27]:
print(df[["smiles", "smiles_std"]].head() )

                                              smiles  \
0                       CCOc1ccc2nc(S(N)(=O)=O)sc2c1   
1                          CCN1C(=O)NC(c2ccccc2)C1=O   
2  CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...   
3                    CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C   
4                          CC(O)(P(=O)(O)O)P(=O)(O)O   

                                          smiles_std  
0                       CCOc1ccc2nc(S(N)(=O)=O)sc2c1  
1                          CCN1C(=O)NC(c2ccccc2)C1=O  
2  CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...  
3                    CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C  
4                          CC(O)(P(=O)(O)O)P(=O)(O)O  


InChI stands for International Chemical Identifier, which is an international identifier for chemical structures. The InChIKey is used here primarily to search for duplicates, not to train an ML model.

In [28]:
from rdkit.Chem import inchi

In [29]:
def mol_to_inchikey(mol) :
    if mol is None:
        return None
    return inchi.MolToInchiKey(mol)        

In [30]:
df["inchikey"] = df["mol"].apply(mol_to_inchikey)

In [31]:
print("unique Inchikeys: ", df["inchikey"].nunique())

unique Inchikeys:  7821


Checking for and removing duplicates

" .duplicated(keep=False)"  checks which values appear more than once. In other words, this line does not delete anything. It simply creates a `duplicate_mask` so that the next step can determine which duplicates to delete.

In [32]:
duplicate_mask = df["inchikey"].duplicated(keep=False)

How many rows in the DataFrame are duplicates?

In [33]:
print("rows belongng to duplicate groups: ", duplicate_mask.sum())

rows belongng to duplicate groups:  4


How many different InChIKey values do the duplicates have?

In [34]:
print("number of duplicate groups:", df.loc[duplicate_mask, "inchikey"].nunique())

number of duplicate groups: 2


In [35]:
print(" rows before duplivate removal:", len(df))

 rows before duplivate removal: 7823


We remove the rows that do not contain an SR-ARE value.
In other words, we will now consider only those molecules that have a label.

In [36]:
sr_are_conflicts = ( df.dropna(subset=["SR-ARE"]).groupby("inchikey")["SR-ARE"].nunique() )                  
                   

Take only those values from `sr_are_conflicts` that are greater than 1, and store them in `conflicting_groups`. Checks for a conflict when the same molecule has two different SR-ARE values.

In [37]:
conflicting_groups = sr_are_conflicts[sr_are_conflicts > 1 ]

In [38]:
print("duplicate groups with conflicting SR-ARE labels: ", len(conflicting_groups) )

duplicate groups with conflicting SR-ARE labels:  0


It retrieves the indices from `conflicting_groups` and stores them in the variable `conflicting_keys`. In this case, the indices are InChIKey.

In [39]:
conflicting_keys=conflicting_groups.index

We identify conflicting molecules

In [40]:
conflicts = ( df[df["inchikey"].isin(conflicting_keys)][["inchikey", "smiles_std", "SR-ARE"]].sort_values("inchikey") )

In [41]:
conflicts

,inchikey,smiles_std,SR-ARE


Remove compounds with a conflicting SR-ARE label

In [42]:
conflict_rows=df[df["inchikey"].isin(conflicting_groups.index)]

In [43]:
print("number of conflicting compounds:", conflicting_groups.shape[0])

number of conflicting compounds: 0


Create a new, cleaned DataFrame

In [44]:
df_clean = df[~df["inchikey"].isin(conflicting_groups.index)].copy()

In [45]:
print("clean rows:", len(df_clean))

clean rows: 7823


Removing Duplicates

In [46]:
df_clean = ( df.sort_values("inchikey").drop_duplicates(subset="inchikey", keep="first").copy() )

In [47]:
print("rows before duplicate removal :", len(df) )

rows before duplicate removal : 7823


In [48]:
print("rows after ducplicate removal:", len(df_clean))

rows after ducplicate removal: 7821


We check the cleaning results

In [49]:
print("clean dataset shape:", df_clean.shape)

clean dataset shape: (7821, 17)


In [50]:
print(f"missing values: {df_clean.isna().sum()}")

missing values: NR-AR             565
NR-AR-LBD        1072
NR-AhR           1281
NR-Aromatase     2008
NR-ER            1637
NR-ER-LBD         875
NR-PPAR-gamma    1380
SR-ARE           1998
SR-ATAD5          758
SR-HSE           1363
SR-MMP           2018
SR-p53           1056
mol_id              0
smiles              0
mol                 0
smiles_std          0
inchikey            0
dtype: int64


Before training the model, you need to remove the rows where SR-ARE is missing, because the model requires a known y.

In [51]:
print(f"SR-ARE distribution: {df_clean['SR-ARE'].value_counts()}")

SR-ARE distribution: SR-ARE
0.0    4881
1.0     942
Name: count, dtype: int64


In [52]:
df_clean_na = df_clean.dropna(subset=["SR-ARE"]).copy()

In [53]:
print(f" SR ARE distribution: {df_clean_na['SR-ARE'].value_counts() }")

 SR ARE distribution: SR-ARE
0.0    4881
1.0     942
Name: count, dtype: int64


In [58]:
output_file= "../output/tox21_clean.csv"

In [61]:
#print(f"clean dataset saved as: {output_file}")

In [62]:
import os

# Create output directory if it does not exist
os.makedirs("../output", exist_ok=True)

# Save cleaned dataset
output_file = "../output/tox21_clean.csv"
df_clean_na.to_csv(output_file, index=False)

print(f"Clean dataset successfully saved: {output_file}")

Clean dataset successfully saved: ../output/tox21_clean.csv


# molecular standardization

In [63]:
df=pd.read_csv("../output/tox21_clean.csv")

In [64]:
df.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol,smiles_std,inchikey
0,0.0,0.0,0.0,1.0,1.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,TOX25769,CCc1cc(Cl)c(OC)c(C(=O)NC[C@@H]2CCCN2CC)c1O,<rdkit.Chem.rdchem.Mol object at 0x0000017A2FF...,CCc1cc(Cl)c(OC)c(C(=O)NC[C@@H]2CCCN2CC)c1O,AADCDMQTJNYOSS-LBPRGKRZSA-N
1,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,TOX25578,O=C(O)[C@H]1[C@@H](O)CC[C@H]2CN3CCc4c([nH]c5cc...,<rdkit.Chem.rdchem.Mol object at 0x0000017A2FF...,O=C(O)[C@@H]1[C@H]2C[C@H]3c4[nH]c5ccccc5c4CCN3...,AADVZSXPNRLYLV-GKMXPDSGSA-N
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX4272,CSc1nc(NC(C)C)nc(NC(C)C)n1,<rdkit.Chem.rdchem.Mol object at 0x0000017A2FF...,CSc1nc(NC(C)C)nc(NC(C)C)n1,AAEVYOVXGOFMJO-UHFFFAOYSA-N
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX5879,O=C(O)C=Cc1ccc(C=CC(=O)O)cc1,<rdkit.Chem.rdchem.Mol object at 0x0000017A2FF...,O=C(O)C=Cc1ccc(C=CC(=O)O)cc1,AAFXQFIGKBLKMC-UHFFFAOYSA-N
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX21446,OC1CCCCC1c1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x0000017A2FF...,OC1CCCCC1c1ccccc1,AAIBYZBZXNWTPP-UHFFFAOYSA-N


let's choose SR-ARE

In [65]:
df_sr_are=df.dropna(subset=["SR-ARE"]).copy()

In [66]:
print("number of labels compunds:", len(df_sr_are))

number of labels compunds: 5823


In [67]:
print(df_sr_are["SR-ARE"].value_counts())

SR-ARE
0.0    4881
1.0     942
Name: count, dtype: int64


prepare SMILES for X values 

In [68]:
smiles=df_sr_are["smiles"].values

In [69]:
smiles

<StringArray>
[                          'CCc1cc(Cl)c(OC)c(C(=O)NC[C@@H]2CCCN2CC)c1O',
 'O=C(O)[C@H]1[C@@H](O)CC[C@H]2CN3CCc4c([nH]c5ccccc45)[C@@H]3C[C@@H]21',
                                           'CSc1nc(NC(C)C)nc(NC(C)C)n1',
                                         'O=C(O)C=Cc1ccc(C=CC(=O)O)cc1',
                                                    'OC1CCCCC1c1ccccc1',
                           'C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1',
                                                        'CCCCCCCCC1CO1',
                                  'CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21',
                    'O=S1(=O)OCC2C(CO1)C1(Cl)C(Cl)=C(Cl)C2(Cl)C1(Cl)Cl',
                                                      'CCOP(=O)(CC)OCC',
 ...
                                                 'Cc1ccc(-c2ccccc2)cc1',
                                          'Cc1c(I)c(=O)n(-c2ccccc2)n1C',
                                'COc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(OC)n1',
                                

for now we have smiles is array of strings, next step is:
Take the existing experimental results from the SR-ARE column and store them in the array y. 
We obtain labels from the experimental data: 0 and 1. Binary classification is encoded  as follows:
0 - negative result
1 - positive result

In [70]:
y=df_sr_are["SR-ARE"].astype(int).values

In [71]:
y[0:10]

array([0, 0, 0, 0, 0, 1, 0, 0, 1, 0])

In [72]:
print(f"number of molecules: {len(smiles)}")
print(f"number of labels: {len(y)}")

number of molecules: 5823
number of labels: 5823


### Morgan Fingerprints
Morgan Fingerprints is a method for converting a molecule described in SMILES into a sequence of 0s and 1s—a long binary vector—so that a machine learning model can process the molecule.

In [73]:
from rdkit import Chem

In [74]:
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

let's create a generator

In [75]:

morgan_generator = GetMorganGenerator(
    radius=2,
    fpSize=2048
)

A function that converts a SMILES string into a Morgan Fingerprint.

Create fingerprints

In [76]:
def smiles_to_morgan(smiles):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None
    
    return morgan_generator.GetFingerprintAsNumPy(mol)


In [77]:
X_list=[]


In [78]:
valid_indices=[]

In [79]:
for i, smi in enumerate(smiles) :
    fp=smiles_to_morgan(smi)
    if fp is not  None :
        X_list.append(fp)
        valid_indices.append(i)

X_list contains Morgan fingerprints.

In [80]:
X_list[:10]

[array([0, 0, 0, ..., 1, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 1], shape=(2048,), dtype=uint8),
 array([0, 1, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8)]

In [81]:
valid_indices[:10]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Previously, X_list was a list of individual fingerprints: np.stack() combines them into a single matrix:

In [82]:
X=np.stack(X_list)

In [83]:
y=y[valid_indices]

In [84]:
print("X shape:", X.shape)
print("y shape: ", y.shape)

X shape: (5823, 2048)
y shape:  (5823,)


TRAIN-TEST split
Previously, wee  had:

X = 5,633 molecules × 2,048 features
y = 5,633 experimental responses

Now we divide them into two groups:

                    ALL DATA
                 5,633 molecules
                       │
             ┌─────────┴─────────┐
             ↓                   ↓
         TRAIN                 TEST
      80% ≈ 4506             20% ≈ 1127

In [85]:
from sklearn.model_selection import train_test_split

In [86]:
X_train, X_test, y_train, y_test = train_test_split (
    X,
    y,
    test_size= 0.2,
    random_state=42,
    stratify=y
)

# Model 1 Logistic Regression

In [87]:
from sklearn.linear_model import LogisticRegression

In [88]:
lr= LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

In [89]:
lr.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

predict_proba() returns two numbers for each molecule:

             class 0     class 1
molecule 1    0.9656       0.0344
molecule 2    0.3023       0.6977
molecule 3    0.9787       0.0213

What Does This Mean?

These are the Class 1 probabilities generated by the Logistic regression for each of the 1,127 test molecules.

In [90]:
y_score_lr= lr.predict_proba(X_test)[:,1]

the probability that a molecule belongs to class 1. For example: 0.0343722 means:

Logistic Regression estimates the probability of class 1 for this molecule to be approximately 3.4%.

In [91]:
y_score_lr

array([0.08262999, 0.81647566, 0.34998724, ..., 0.81731354, 0.87647142,
       0.24604993], shape=(1165,))

# ROC AUC

In [92]:
from sklearn.metrics import roc_auc_score

In [93]:
roc_lr=roc_auc_score(
    y_test,
    y_score_lr
)

In [94]:
print(f"logistic regression ROC-AUC:  {roc_lr} ")

logistic regression ROC-AUC:  0.7950684901674687 


# Model 2 Random Forest 


In [95]:
from sklearn.ensemble import RandomForestClassifier

Here, we create a Random Forest model for classification:
We create a model object and name it rf.

A Random Forest consists of many decision trees that collectively determine the class.

For example, we have:

0 = the molecule is inactive
1 = the molecule is active

then the Random Forest will learn to predict 0 or 1.

In [101]:
rf=RandomForestClassifier(
    n_estimators= 300,
    class_weight="balanced",
    random_state=42,
    n_jobs=1 )

In [102]:
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [103]:
type(rf)

sklearn.ensemble._forest.RandomForestClassifier

These are the Class 1 probabilities generated by the Random Forest model for each of the 1,127 test molecules.

In [104]:
y_score_rf = rf.predict_proba(X_test)[:, 1]

In [105]:
y_score_rf

array([0.04666667, 0.37666667, 0.29666667, ..., 0.35      , 0.23333333,
       0.29      ], shape=(1165,))

roc_auc_score()

This is a function from sklearn that evaluates how well a model distinguishes between classes 0 and 1.


In [106]:

roc_rf = roc_auc_score (
    y_test,
    y_score_rf)

In [107]:
print(f"random forest  ROC-AUC {roc_rf}")
print(f"logitic regression ROC-AUC {roc_lr}")

random forest  ROC-AUC 0.8046233585226159
logitic regression ROC-AUC 0.7950684901674687


# Metrics. F1, Precision and Recall

Here, we make the standard binary predictions for both models.

In [108]:
y_pred_lr=lr.predict(X_test)
y_pred_rf=rf.predict(X_test)

In [109]:
y_pred_lr

array([0, 1, 0, ..., 1, 1, 0], shape=(1165,))

In [110]:
y_pred_rf

array([0, 0, 0, ..., 0, 0, 0], shape=(1165,))

In [111]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

## for logistic regression: 

In [112]:
print(f"logistic regression:")
print(f"precision: {precision_score(y_test,y_pred_lr)}")
print(f"recall: {recall_score(y_test, y_pred_lr)}  ")
print(f"F1: {f1_score(y_test, y_pred_lr)}")
print(f"accuracy : {accuracy_score(y_test, y_pred_lr)}")

logistic regression:
precision: 0.375
recall: 0.574468085106383  
F1: 0.453781512605042
accuracy : 0.776824034334764


## for ranadom forest 

In [113]:
print("RANDOM FOREST")

print("Precision:",  precision_score(y_test, y_pred_rf))

print("Recall:",  recall_score(y_test, y_pred_rf))

print("F1:", f1_score(y_test, y_pred_rf))

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

RANDOM FOREST
Precision: 0.7924528301886793
Recall: 0.22340425531914893
F1: 0.34854771784232363
Accuracy: 0.8652360515021459


## summary table 

In [114]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_score_lr),
        roc_auc_score(y_test, y_score_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    "F1": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ]
})

results

,Model,ROC-AUC,Precision,Recall,F1,Accuracy
0,Logistic Regression,0.795068,0.375000,0.574468,0.453782,0.776824
1,Random Forest,0.804623,0.792453,0.223404,0.348548,0.865236
